# CSIRO Biomass - Teacher Distillation Training

Train an image model using true biomass targets plus soft targets from a metadata teacher trained inside each fold. This is intended as the second GPU run after multitask NDVI/height learning.

In [ ]:
import os, sys, json, random, subprocess, pkgutil
from pathlib import Path
import numpy as np
import pandas as pd

if pkgutil.find_loader('timm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
if pkgutil.find_loader('lightgbm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'])

import cv2, torch, timm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OrdinalEncoder
from lightgbm import LGBMRegressor

DATA_DIR = Path('/kaggle/input/csiro-biomass')
OUT_DIR = Path('/kaggle/working')
CKPT_DIR = OUT_DIR / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda': raise RuntimeError('GPU is required for this experiment.')

SEED = 43; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
IMAGE_SIZE = int(os.getenv('IMAGE_SIZE', '384'))
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '16'))
EPOCHS = int(os.getenv('EPOCHS', '10'))
N_FOLDS = int(os.getenv('N_FOLDS', '5'))
BACKBONE = os.getenv('BACKBONE', 'convnextv2_tiny.fcmae_ft_in22k_in1k')
DISTILL_WEIGHT = float(os.getenv('DISTILL_WEIGHT', '0.35'))
TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
WEIGHTS = np.array([0.1, 0.1, 0.1, 0.2, 0.5], dtype=np.float32)

In [ ]:
train_long = pd.read_csv(DATA_DIR / 'train.csv')
meta_cols = [c for c in ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'] if c in train_long.columns]
meta = train_long[['image_path'] + [c for c in meta_cols if c != 'image_path']].drop_duplicates('image_path')
wide = train_long.pivot_table(index='image_path', columns='target_name', values='target', aggfunc='first').reset_index()
df = meta.merge(wide, on='image_path', how='left')
df['image_id'] = df['image_path'].astype(str)
if 'Sampling_Date' in df.columns:
    dt = pd.to_datetime(df['Sampling_Date'], errors='coerce')
    df['month'] = dt.dt.month.fillna(0).astype(int)
    df['dayofyear'] = dt.dt.dayofyear.fillna(0).astype(int)
cat_cols = [c for c in ['State', 'Species'] if c in df.columns]
num_cols = [c for c in ['Pre_GSHH_NDVI', 'Height_Ave_cm', 'month', 'dayofyear'] if c in df.columns]
print(df.shape, cat_cols, num_cols)

In [ ]:
def make_meta_features(train_part, valid_part):
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    xtr = train_part[num_cols].copy() if num_cols else pd.DataFrame(index=train_part.index)
    xva = valid_part[num_cols].copy() if num_cols else pd.DataFrame(index=valid_part.index)
    if cat_cols:
        trc = pd.DataFrame(enc.fit_transform(train_part[cat_cols].fillna('missing')), columns=cat_cols, index=train_part.index)
        vac = pd.DataFrame(enc.transform(valid_part[cat_cols].fillna('missing')), columns=cat_cols, index=valid_part.index)
        xtr = pd.concat([xtr, trc], axis=1); xva = pd.concat([xva, vac], axis=1)
    return xtr.fillna(-999), xva.fillna(-999)

class DistillDataset(Dataset):
    def __init__(self, frame, teacher, train=True):
        self.df = frame.reset_index(drop=True); self.teacher = teacher.astype(np.float32); self.train = train
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(str(DATA_DIR / row.image_path)); image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.train and random.random() < 0.5: image = np.ascontiguousarray(image[:, ::-1])
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE)).astype(np.float32) / 255.0
        image = (image - np.array([0.485,0.456,0.406], np.float32)) / np.array([0.229,0.224,0.225], np.float32)
        return torch.from_numpy(image.transpose(2,0,1)).float(), torch.tensor(row[TARGETS].values.astype(np.float32)), torch.tensor(self.teacher[idx])

class ImageRegressor(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder = timm.create_model(BACKBONE, pretrained=True, num_classes=0, global_pool='avg')
        self.head = nn.Sequential(nn.LayerNorm(self.encoder.num_features), nn.Dropout(0.15), nn.Linear(self.encoder.num_features, 512), nn.GELU(), nn.Linear(512, len(TARGETS)))
    def forward(self, x): return self.head(self.encoder(x))

def weighted_r2(y, p):
    vals = [1 - np.sum((y[:, i]-p[:, i])**2) / max(np.sum((y[:, i]-y[:, i].mean())**2), 1e-12) for i in range(len(TARGETS))]
    return float(np.sum(np.array(vals) * WEIGHTS)), dict(zip(TARGETS, vals))

def loss_fn(pred, y, teacher):
    w = torch.tensor(WEIGHTS, device=pred.device).view(1, -1)
    hard = ((pred - y) ** 2 * w).mean()
    soft = ((pred - teacher) ** 2 * w).mean()
    return hard + DISTILL_WEIGHT * soft

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)
oof = np.zeros((len(df), len(TARGETS)), np.float32); teacher_oof = np.zeros_like(oof); logs = []
for fold, (trn_idx, val_idx) in enumerate(gkf.split(df, groups=df.image_id)):
    trn, val = df.iloc[trn_idx], df.iloc[val_idx]
    xtr, xva = make_meta_features(trn, val)
    teacher = np.zeros((len(val), len(TARGETS)), np.float32)
    teacher_tr = np.zeros((len(trn), len(TARGETS)), np.float32)
    for i, t in enumerate(TARGETS):
        m = LGBMRegressor(n_estimators=600, learning_rate=0.025, num_leaves=16, random_state=SEED+i, verbose=-1)
        m.fit(xtr, trn[t])
        teacher[:, i] = m.predict(xva); teacher_tr[:, i] = m.predict(xtr)
    teacher_oof[val_idx] = teacher
    trn_dl = DataLoader(DistillDataset(trn, teacher_tr, True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_dl = DataLoader(DistillDataset(val, teacher, False), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
    model = ImageRegressor().to(DEVICE); opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4); scaler = GradScaler()
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS); best = -999; best_pred = None
    for epoch in range(EPOCHS):
        model.train()
        for x, y, tea in trn_dl:
            x, y, tea = x.to(DEVICE), y.to(DEVICE), tea.to(DEVICE); opt.zero_grad(set_to_none=True)
            with autocast(): loss = loss_fn(model(x), y, tea)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step(); model.eval(); preds = []
        with torch.no_grad():
            for x, y, tea in val_dl:
                with autocast(): preds.append(model(x.to(DEVICE)).float().cpu().numpy())
        pred = np.concatenate(preds); score, detail = weighted_r2(val[TARGETS].values.astype(np.float32), pred)
        logs.append({'fold': fold, 'epoch': epoch, 'weighted_r2': score, **detail})
        print(f'fold={fold} epoch={epoch} weighted_r2={score:.6f}')
        if score > best:
            best = score; best_pred = pred.copy(); torch.save({'model': model.state_dict(), 'backbone': BACKBONE, 'image_size': IMAGE_SIZE}, CKPT_DIR / f'fold{fold}.pt')
    oof[val_idx] = best_pred
score, detail = weighted_r2(df[TARGETS].values.astype(np.float32), oof)
print('OOF', score, detail)

In [ ]:
out = df[['image_path']].copy()
for i, t in enumerate(TARGETS):
    out[f'pred_{t}'] = oof[:, i]; out[f'teacher_{t}'] = teacher_oof[:, i]; out[f'true_{t}'] = df[t].values
out.to_csv(OUT_DIR / 'oof_predictions.csv', index=False)
pd.DataFrame(logs).to_csv(OUT_DIR / 'training_log.csv', index=False)
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump({'cv_weighted_r2': score, 'per_target_r2': detail, 'distill_weight': DISTILL_WEIGHT, 'backbone': BACKBONE}, f, indent=2)
print('Saved distillation artifacts. Use kaggle_submission.ipynb or this run checkpoint outputs for inference.')